In [14]:
import pandas as pd
from scipy.stats import variation, normaltest, mannwhitneyu
import plotly.express as px
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind

In [4]:
tvs_per_wb_df = pd.read_csv(r"X:\pervasive_group\Shared\MobiliseD TVS\Sharepoint dataset\TVS-Final-DMO-Dataset\tvs-wb-dmo-17-05-2023.csv")

metadata = pd.read_csv(r"X:\pervasive_group\Shared\MobiliseD TVS\Sharepoint dataset\TVS-Final-Clinical-Dataset\study-instances-Cohort Site-2023-03-01h15m31s39.csv")

# get HA ids from metadata file
HA_participant_ids = metadata[metadata["Cohort HA"] == "T"]["Local Participant"]

# trim wb file to only HA participants
tvs_per_wb_df = tvs_per_wb_df[tvs_per_wb_df["participantid"].isin(HA_participant_ids)]

In [5]:
tvs_daily_dmos = pd.DataFrame()

for participant in tvs_per_wb_df["participantid"].unique():
    per_wb_participant_df = tvs_per_wb_df[tvs_per_wb_df["participantid"] == participant]

    #calculate aggregate DMOs per day
    for day in per_wb_participant_df["wbday"].unique():
        per_day_wb_participant_df = per_wb_participant_df[per_wb_participant_df["wbday"] == day]

        # get subset wb durations
        wb_10 = per_day_wb_participant_df[per_day_wb_participant_df["duration"] > 10]
        wb_10_30 = per_day_wb_participant_df[(per_day_wb_participant_df["duration"] > 10) & (per_day_wb_participant_df["duration"] < 30)]
        wb_30 = per_day_wb_participant_df[per_day_wb_participant_df["duration"] > 30]
        wb_60 = per_day_wb_participant_df[per_day_wb_participant_df["duration"] > 60]

        daily_level_dmos_df = pd.DataFrame(index=[0])
        daily_level_dmos_df["participant_id"] = participant
        daily_level_dmos_df["day"] = day

        # amount
        daily_level_dmos_df["walkdur_all_sum_d"] = per_day_wb_participant_df["duration"].sum()/60
        daily_level_dmos_df["steps_all_sum_d"] = per_day_wb_participant_df["numberstrides"].sum() # should be steps but number of steps aren't included in the per-wb DMOs?

        # pattern
        daily_level_dmos_df["wb_all_sum_d"] = len(per_day_wb_participant_df)
        daily_level_dmos_df["wb_10_sum_d"] = len(wb_10)
        daily_level_dmos_df["wb_30_sum_d"] = len(wb_30)
        daily_level_dmos_df["wb_60_sum_d"] = len(wb_60)
        daily_level_dmos_df["wbdur_all_avg_d"] = per_day_wb_participant_df["duration"].median()
        daily_level_dmos_df["wbdur_all_p90_d"] = per_day_wb_participant_df["duration"].quantile(0.9)
        daily_level_dmos_df["wbdur_all_var_d"] = variation(per_day_wb_participant_df["duration"])

        # pace
        daily_level_dmos_df["ws_1030_avg_d"] = wb_10_30["averagestridespeed"].mean()
        daily_level_dmos_df["ws_30_avg_d"] = wb_30["averagestridespeed"].mean()
        daily_level_dmos_df["ws_10_p90_d"] = wb_10["averagestridespeed"].quantile(0.9)
        daily_level_dmos_df["ws_30_p90_d"] = wb_30["averagestridespeed"].quantile(0.9)
        daily_level_dmos_df["strlen_1030_avg_d"] = wb_10_30["averagestridelength"].mean()
        daily_level_dmos_df["strlen_30_avg_d"] = wb_30["averagestridelength"].mean()

        # rhythm
        daily_level_dmos_df["cadence_all_avg_d"] = per_day_wb_participant_df["averagecadence"].mean()
        daily_level_dmos_df["cadence_30_avg_d"] = wb_30["averagecadence"].mean()
        daily_level_dmos_df["cadence_30_p90_d"] = wb_30["averagecadence"].quantile(0.9)
        daily_level_dmos_df["strdur_all_avg_d"] = per_day_wb_participant_df["averagestrideduration"].mean()
        daily_level_dmos_df["strdur_30_avg_d"] = wb_30["averagestrideduration"].mean()

        # bout to bout variability
        daily_level_dmos_df["ws_30_var_d"] = variation(wb_30["averagestridespeed"])
        daily_level_dmos_df["strlen_30_var_d"] = variation(wb_30["averagestridelength"])
        daily_level_dmos_df["cadence_all_var_d"] = variation(per_day_wb_participant_df["averagecadence"])
        daily_level_dmos_df["strdur_all_var_d"] = variation(per_day_wb_participant_df["averagestrideduration"])

        tvs_daily_dmos = pd.concat([tvs_daily_dmos, daily_level_dmos_df], ignore_index=True)

tvs_weekly_dmos = pd.DataFrame()
for participant in tvs_daily_dmos["participant_id"].unique():
    all_days_dmos_df = tvs_daily_dmos[tvs_daily_dmos["participant_id"] == participant].filter(regex="_d$").mean().to_frame().T
    all_days_dmos_df["participant_id"] = participant
    tvs_weekly_dmos = pd.concat([tvs_weekly_dmos, all_days_dmos_df], ignore_index=True)

#print(tvs_daily_dmos)
#print(tvs_weekly_dmos)

C:\Users\ac4jmi\AppData\Local\Temp\ipykernel_27796\1507662471.py:49: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  daily_level_dmos_df["ws_30_var_d"] = variation(wb_30["averagestridespeed"])
C:\Users\ac4jmi\AppData\Local\Temp\ipykernel_27796\1507662471.py:50: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  daily_level_dmos_df["strlen_30_var_d"] = variation(wb_30["averagestridelength"])
C:\Users\ac4jmi\AppData\Local\Temp\ipykernel_27796\1507662471.py:49: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  daily_level_dmos_df["ws_30_var_d"] = variation(wb_30["averagestridespeed"])
C:\Users\ac4jmi\AppData\Local\Temp\ipykernel_27796\1507662471.py:50: SmallSampleWarning: One or more sample arguments is t

In [6]:
dmo4lnc_per_wb_df = pd.read_csv("Free-Living Parameters/per_wb_parameters.csv")
dmo4lnc_daily_dmos = pd.read_csv("Free-Living Parameters/all_day_parameters.csv")
dmo4lnc_weekly_dmos = pd.read_csv("Free-Living Parameters/weekly_parameters.csv")

dmo4lnc_per_wb_cp_df = dmo4lnc_per_wb_df[dmo4lnc_per_wb_df["cohort"] == "CP"]
dmo4lnc_daily_cp_dmos = dmo4lnc_daily_dmos[dmo4lnc_daily_dmos["cohort"] == "CP"]
dmo4lnc_weekly_cp_dmos = dmo4lnc_weekly_dmos[dmo4lnc_weekly_dmos["cohort"] == "CP"]

dmo4lnc_per_wb_cp_df.to_csv("HA_vs_CP/CP_per_wb.csv")
dmo4lnc_daily_cp_dmos.to_csv("HA_vs_CP/CP_daily.csv")
dmo4lnc_weekly_dmos.to_csv("HA_vs_CP/CP_weekly.csv")

tvs_per_wb_df.to_csv("HA_vs_CP/HA_per_wb.csv")
tvs_daily_dmos.to_csv("HA_vs_CP/HA_daily.csv")
tvs_weekly_dmos.to_csv("HA_vs_CP/HA_weekly.csv")


In [17]:
def calculate_IQR(df):
    # Calculating Q1 and Q3
    Q1 = df.quantile(0.25)
    Q3 = df.quantile(0.75)

    # Calculating IQR
    return Q3 - Q1

def do_stats(cp_s, ha_s, name):
    print()
    print("*"*50)
    print(name, ha_s.name)
    print("*"*50)
    print()
    print("Normal test:")
    enough_data_cp = len(cp_s) >= 8
    enough_data_ha = len(ha_s) >= 8
    use_normal = False
    if normaltest(cp_s, nan_policy='omit') and enough_data_cp and normaltest(ha_s, nan_policy='omit') and enough_data_ha:
        print(f"{cp_s.name} is normally distributed")
        print(f"{ha_s.name} is normally distributed")
        use_normal = True
    elif not enough_data_cp or not enough_data_ha:
        print(f"not enough data in cp or ha to determine normality")
        use_normal = False

    if use_normal:
        print()
        print(f"mean DMO4LNC CP: {cp_s.mean()}")
        print(f"mean TVS HA: {ha_s.mean()}")

        print()
        print(f"standard deviation DMO4LNC CP: {cp_s.std()}")
        print(f"standard deviation TVS HA: {ha_s.std()}")

        t_test_results = ttest_ind(cp_s.dropna(), ha_s.dropna())
        print()
        print("T-Test of CP vs HA:")
        print(f"statistic: {t_test_results.statistic}, p-value: {t_test_results.pvalue}")
    else:
        print()
        print(f"median DMO4LNC CP: {cp_s.median()}")
        print(f"median TVS HA: {ha_s.median()}")

        print()
        print(f"IQR DMO4LNC CP: {calculate_IQR(cp_s)}")
        print(f"IQR DMO4LNC HA: {calculate_IQR(ha_s)}")

        mannwhitney_results = mannwhitneyu(cp_s, ha_s, nan_policy='omit')
        print()
        print("Mann-Whitney test of CP vs HA:")
        print(f"statistic: {mannwhitney_results.statistic}, p-value: {mannwhitney_results.pvalue}")


do_stats(dmo4lnc_per_wb_cp_df["cadence_spm"], tvs_per_wb_df["averagecadence"], "per-wb")
do_stats(dmo4lnc_per_wb_cp_df["duration_s"], tvs_per_wb_df["duration"], "per-wb")

do_stats(dmo4lnc_weekly_cp_dmos["wb_30__count_w"], tvs_weekly_dmos["wb_30_sum_d"], "weekly")
do_stats(dmo4lnc_daily_cp_dmos["wb_30__count"], tvs_daily_dmos["wb_30_sum_d"], "daily")

do_stats(dmo4lnc_weekly_cp_dmos["wb_60__count_w"], tvs_weekly_dmos["wb_60_sum_d"], "weekly")
do_stats(dmo4lnc_daily_cp_dmos["wb_60__count"], tvs_daily_dmos["wb_60_sum_d"], "daily")

do_stats(dmo4lnc_weekly_cp_dmos["wb_all__count_w"], tvs_weekly_dmos["wb_all_sum_d"], "weekly")
do_stats(dmo4lnc_daily_cp_dmos["wb_all__count"], tvs_daily_dmos["wb_all_sum_d"], "daily")

do_stats(dmo4lnc_weekly_cp_dmos["wb_all__cadence_spm__avg_w"], tvs_weekly_dmos["cadence_all_avg_d"], "weekly")
do_stats(dmo4lnc_daily_cp_dmos["wb_all__cadence_spm__avg"], tvs_daily_dmos["cadence_all_avg_d"], "daily")

do_stats(dmo4lnc_weekly_cp_dmos["wb_30__cadence_spm__avg_w"], tvs_weekly_dmos["cadence_30_avg_d"], "weekly")
do_stats(dmo4lnc_daily_cp_dmos["wb_30__cadence_spm__avg"], tvs_daily_dmos["cadence_30_avg_d"], "daily")

do_stats(dmo4lnc_weekly_cp_dmos["total_walking_duration_min_w"], tvs_weekly_dmos["walkdur_all_sum_d"], "weekly")
do_stats(dmo4lnc_daily_cp_dmos["total_walking_duration_min"], tvs_daily_dmos["walkdur_all_sum_d"], "daily")

do_stats(dmo4lnc_weekly_dmos["wb_30__walking_speed_mps__avg_w"], tvs_weekly_dmos["ws_30_avg_d"], "weekly")
do_stats(dmo4lnc_daily_dmos["wb_30__walking_speed_mps__avg"], tvs_weekly_dmos["ws_30_avg_d"], "daily")

do_stats(dmo4lnc_weekly_dmos["wb_30__stride_length_m__avg_w"], tvs_weekly_dmos["strlen_30_avg_d"], "weekly")
do_stats(dmo4lnc_daily_dmos["wb_30__stride_length_m__avg"], tvs_weekly_dmos["strlen_30_avg_d"], "daily")
#plt.figure()
#plt.hist(tvs_per_wb_df["averagecadence"], bins=100)
#plt.hist(dmo4lnc_per_wb_cp_df["cadence_spm"], bins=100)
#plt.show()


**************************************************
per-wb averagecadence
**************************************************

Normal test:
cadence_spm is normally distributed
averagecadence is normally distributed

mean DMO4LNC CP: 86.27512722528007
mean TVS HA: 85.4292842796705

standard deviation DMO4LNC CP: 10.025180381389719
standard deviation TVS HA: 11.428162188775657

T-Test of CP vs HA:
statistic: 7.371660386424456, p-value: 1.709791403753015e-13

**************************************************
per-wb duration
**************************************************

Normal test:
duration_s is normally distributed
duration is normally distributed

mean DMO4LNC CP: 19.21269695979078
mean TVS HA: 21.250250803063974

standard deviation DMO4LNC CP: 31.24832741933482
standard deviation TVS HA: 112.02494942698728

T-Test of CP vs HA:
statistic: -1.9886678219663811, p-value: 0.04674304239216348

**************************************************
weekly wb_30_sum_d
***********************

C:\Users\ac4jmi\AppData\Local\Temp\ipykernel_27796\2052467325.py:19: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  if normaltest(cp_s, nan_policy='omit') and enough_data_cp and normaltest(ha_s, nan_policy='omit') and enough_data_ha:
C:\Users\ac4jmi\AppData\Local\Temp\ipykernel_27796\2052467325.py:19: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  if normaltest(cp_s, nan_policy='omit') and enough_data_cp and normaltest(ha_s, nan_policy='omit') and enough_data_ha:
C:\Users\ac4jmi\AppData\Local\Temp\ipykernel_27796\2052467325.py:19: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  if normaltest(cp_s, nan_policy='omit') and enough_data_cp and normaltest(ha_s, nan_policy='omit') and enough_data_ha:
C

In [9]:
# 1. Calculate means and create a dictionary
data = {
    "Metric": ['Avg Cadence', 'Walk Duration', 'Walking Count'] * 2,
    "Value": [
        # Cohort 1 Means
        dmo4lnc_weekly_cp_dmos["wb_all__cadence_spm__avg_w"].mean(),
        dmo4lnc_weekly_cp_dmos["total_walking_duration_min_w"].mean(),
        dmo4lnc_weekly_cp_dmos["wb_all__count_w"].mean(),
        # Cohort 2 Means
        tvs_weekly_dmos["cadence_all_avg_d"].mean(),
        tvs_weekly_dmos["walkdur_all_sum_d"].mean(),
        tvs_weekly_dmos["wb_all_sum_d"].mean()
    ],
    "Cohort": ['CP'] * 3 + ['HA'] * 3
}

df_plot = pd.DataFrame(data)

# 2. Create the Radar Plot
fig = px.line_polar(
    df_plot,
    r='Value',
    theta='Metric',
    color='Cohort',
    line_close=True,
    title="Cohort Comparison"
)

# 3. Optional: Fill the area under the lines
fig.update_traces(fill='toself')

fig.show()